*0.2 Math / ML basics*

# loss functions (cross-entropy)

**The situation.** The ticket router predicts a team with a probability for each. To train it, you need a number that says how bad each prediction was — one that punishes "90% technical" when the answer was billing far more than "40% technical". Mean squared error does not care enough about confident mistakes.

**Cross-entropy.** Take the probability the model gave to the *correct* answer, and compute minus its log. Right answer at 100% → loss 0. Right answer at 50% → 0.69. Right answer at 1% → 4.6. Confident and wrong is punished hardest. This is the loss every classifier uses, and — because "predict the next token" is a classification over the vocabulary — the loss every language model is trained with.

In [1]:
# Load OPENAI_API_KEY from the .env file. The OpenAI clients read it from the environment.
from dotenv import find_dotenv, load_dotenv

load_dotenv(find_dotenv())
MODEL = "gpt-4o-mini"

In [2]:
import torch
import torch.nn.functional as F

teams = ["billing", "technical", "sales"]
correct = torch.tensor([0])  # the answer is billing
cases = {
    "confident and right": torch.tensor([[4.0, 0.5, 0.0]]),
    "unsure": torch.tensor([[1.0, 0.8, 0.6]]),
    "confident and wrong": torch.tensor([[0.0, 4.0, 0.5]]),
}
for label, logits in cases.items():
    probability_of_billing = torch.softmax(logits, dim=1)[0, 0]
    loss = F.cross_entropy(logits, correct)  # softmax + minus-log, done stably in one call
    print(f"{label:<22} P(billing) = {probability_of_billing:>5.1%}   loss = {loss.item():.2f}")
assert F.cross_entropy(cases["confident and wrong"], correct) > F.cross_entropy(
    cases["unsure"], correct
)

confident and right    P(billing) = 95.4%   loss = 0.05
unsure                 P(billing) = 40.2%   loss = 0.91
confident and wrong    P(billing) =  1.7%   loss = 4.05


**Reading the output.** Confident and right: loss near 0. Unsure: around 1. Confident and wrong: over 4. The loss grows steeply as the probability on the truth shrinks — exactly the pressure you want during training.

**The same loss is a language model's training loss.** A model's score over the next token is a classification with ~100,000 classes. The reported training loss (say 2.1) is this number averaged over tokens.

In [3]:
vocabulary_size = 100_000
torch.manual_seed(0)
next_token_logits = torch.randn(1, vocabulary_size)
true_next_token = torch.tensor([4242])
loss = F.cross_entropy(next_token_logits, true_next_token)
print(
    "untrained model, one token: loss",
    round(loss.item(), 2),
    "≈ ln(100,000) =",
    round(torch.log(torch.tensor(float(vocabulary_size))).item(), 2),
)
print("meaning: it is guessing uniformly across the vocabulary")
assert abs(loss.item() - 11.5) < 1.5

untrained model, one token: loss 12.33 ≈ ln(100,000) = 11.51
meaning: it is guessing uniformly across the vocabulary


```
P(correct)   1.0    0.5    0.1    0.01
loss         0.0    0.69   2.3    4.6        ← −log P(correct)
```

**The rule to remember.** Cross-entropy = −log(probability given to the right answer). Small when confident and right, huge when confident and wrong. A training loss of 2 means the model gives the true next token about e⁻² ≈ 13% on average.

| Use it when | Don't when | Instead use |
|---|---|---|
| any "pick one class" problem, including next-token prediction | predicting a number (latency, price) | mean squared error |

**Watch out**
- `F.cross_entropy` takes raw logits. Passing softmaxed probabilities applies softmax twice and trains a worse model with no error message.
- Class imbalance: if 95% of tickets are billing, a model that always says billing gets a low loss. Use `weight=` or rebalance.
- Perplexity, the number in model papers, is `exp(cross-entropy)`. Loss 2.1 → perplexity 8.2.